# Calibration

Summer4 port of `tweak_cases_calib` `notebooks/04-Calibration.ipynb`.

Same density as `tb_macro.calibration.make_log_likelihood` (seven terms +
Canberra mixing distance), uniform priors on every name in `PARAM_BOUNDS`,
NUTS tree depth 5, four parallel chains, `PRNGKey(2)`, started at
`get_nuts_init_values()`. Warmup and samples use `N_RUNS_LOCAL` (40), matching
the summer3 local notebook. The longer 100/100 run stays in
`notebooks/05-Calibration.ipynb`.

Corrected default model + folded spline. Not `BayesianModel` — summer4 stock
likelihoods do not cover the log/logit/Canberra terms yet.

Restart the kernel and run from the top so the device-count cell runs before
JAX imports. Use **tb-macro-summer4 (pixi)**.


## Four CPU devices


In [ ]:
import os

flags = os.environ.get("XLA_FLAGS", "").split()
kept = [
    flag
    for flag in flags
    if not flag.startswith("--xla_force_host_platform_device_count=")
]
os.environ["XLA_FLAGS"] = " ".join(["--xla_force_host_platform_device_count=4", *kept])

import jax

if jax.local_device_count() < 4:
    raise RuntimeError(
        f"JAX has {jax.local_device_count()} device(s). Restart the kernel so this "
        "cell can set the device count before JAX is imported."
    )

from datetime import UTC, datetime
from pathlib import Path

import arviz as az
import numpyro
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer

from tb_macro.calibration import (
    MCMC_EXTRA_FIELDS,
    build_mixing_cases,
    compile_case,
    inference_data_from_mcmc,
    make_log_likelihood,
)
from tb_macro.constants import LOCAL_OUTPUT_PATH, N_RUNS_LOCAL
from tb_macro.parameters import BASE_PARAMS, PARAM_BOUNDS, get_nuts_init_values
from tb_macro.utils import write_run_log

print(f"devices {jax.local_device_count()}  N_RUNS_LOCAL {N_RUNS_LOCAL}")


## Likelihood


In [ ]:
case = next(item for item in build_mixing_cases() if item.name == "spline")
compiled = compile_case(case)
evaluate = make_log_likelihood(compiled, mixing=case.mixing, fold=case.fold)
print(case.name)


## Sample


In [ ]:
priors = {name: dist.Uniform(low, high) for name, (low, high) in PARAM_BOUNDS.items()}
for name, (low, high) in PARAM_BOUNDS.items():
    print(f"{name:28s}  Uniform({low}, {high})")


def model() -> None:
    sampled = {name: numpyro.sample(name, prior) for name, prior in priors.items()}
    params = dict(BASE_PARAMS)
    params.update(sampled)
    numpyro.factor("ll", evaluate(params)["total"])


kernel = infer.NUTS(
    model,
    max_tree_depth=5,
    init_strategy=infer.init_to_value(values=get_nuts_init_values()),
)
mcmc = infer.MCMC(
    kernel,
    num_warmup=N_RUNS_LOCAL,
    num_samples=N_RUNS_LOCAL,
    num_chains=4,
    chain_method="parallel",
    progress_bar=True,
)
if mcmc.chain_method != "parallel":
    raise RuntimeError(
        f"Refusing to run with chain_method={mcmc.chain_method!r} on "
        f"{jax.local_device_count()} devices."
    )
print(f"chain_method {mcmc.chain_method}")
mcmc.run(PRNGKey(2), extra_fields=MCMC_EXTRA_FIELDS)


## Posterior


In [ ]:
idata = inference_data_from_mcmc(mcmc)
LOCAL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
nc_path = LOCAL_OUTPUT_PATH / f"{timestamp}.nc"
idata.to_netcdf(nc_path, engine="h5netcdf")
write_run_log(
    LOCAL_OUTPUT_PATH / f"{timestamp}.log",
    extra={
        "idata": nc_path.name,
        "mixing": case.name,
        "match_golden": False,
        "entry": "smooth",
        "num_warmup": N_RUNS_LOCAL,
        "num_samples": N_RUNS_LOCAL,
        "num_chains": 4,
    },
)
print(nc_path)
az.summary(idata)
